# Lab 09 — Deploy the Gemini Multi-Agent System with Streamlit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tertiarycourses/TGS-2020503207-AI-Vibe-Coding-for-Multi-Agents-System/blob/main/labs/lab-09-deploy-the-gemini-multi-agent-system-with-streamlit/lab-09-deploy-the-gemini-multi-agent-system-with-streamlit.ipynb)

**Topic:** 4 — Multi-Agent System Development with Gemini Agent SDK

**Objective:** Deploy a Gemini-based collaborative multi-agent system with Streamlit

Give the Gemini agent team the same web interface treatment, then compare the two deployments side by side and record which ecosystem fits which kind of work.

Full step-by-step instructions are in the Learner Guide.


> **Streamlit does not render inside Colab.** A Streamlit app is a server process driven by `streamlit run`, not a notebook widget — nothing will display in the output cell. This notebook therefore writes the app files to disk with `%%writefile`, and you run them **locally** with the command shown at the end. Treat the notebook as the place you assemble and read the code; treat your own machine as the place you run it.


In [ ]:
!pip install -q google-adk python-dotenv streamlit


In [ ]:
# API keys: prefer Colab Secrets (key icon in the left sidebar).
# Add each secret there, enable notebook access, then run this cell.
# NEVER paste a key into the notebook - a saved notebook keeps it forever.
import os
from getpass import getpass

try:
    from google.colab import userdata  # available in Colab only
except ImportError:
    userdata = None


def set_key(name: str) -> None:
    """Read a secret from Colab Secrets, falling back to a hidden prompt."""
    if os.environ.get(name):
        return
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass(f"Enter {name}: ")
    os.environ[name] = value


set_key("GOOGLE_API_KEY")

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
print("Keys set:", [v for v in ["GOOGLE_API_KEY"] if os.environ.get(v)])


## 1. Write the Gemini team to gemini_team.py

The app imports `build_coordinator`, `APP_NAME` and `USER_ID` from this module. The **factory** matters here: switching model variants means rebuilding the agents, and a factory avoids the "an agent can only have one parent" error.


In [ ]:
%%writefile gemini_team.py
"""The Lab 08 Gemini team: a coordinator over weather and time specialists."""

from datetime import datetime
from zoneinfo import ZoneInfo

from dotenv import load_dotenv
from google.adk.agents import Agent

load_dotenv()

MODEL = "gemini-2.0-flash"
APP_NAME = "multi_agent_lab"
USER_ID = "learner-1"


def get_weather(city: str) -> dict:
    """Retrieve the current weather report for a specified city.

    Args:
        city: The name of the city, for example "Singapore".

    Returns:
        A dict with a 'status' key and either a 'report' or 'error_message'.
    """
    readings = {
        "singapore": "31degC with thundery showers and 84% humidity.",
        "london": "12degC and overcast.",
        "tokyo": "18degC and clear.",
    }
    report = readings.get(city.strip().lower())
    if report is None:
        return {
            "status": "error",
            "error_message": f"No weather data available for '{city}'.",
        }
    return {"status": "success", "report": f"The weather in {city} is {report}"}


def get_time(city: str) -> dict:
    """Return the current local time in a specified city.

    Args:
        city: The name of the city, for example "Singapore".

    Returns:
        A dict with a 'status' key and either a 'report' or 'error_message'.
    """
    zones = {
        "singapore": "Asia/Singapore",
        "london": "Europe/London",
        "tokyo": "Asia/Tokyo",
    }
    zone = zones.get(city.strip().lower())
    if zone is None:
        return {
            "status": "error",
            "error_message": f"No timezone information for '{city}'.",
        }
    now = datetime.now(ZoneInfo(zone))
    return {
        "status": "success",
        "report": f"The current time in {city} is {now:%Y-%m-%d %H:%M:%S %Z}.",
    }


def build_coordinator(model: str = MODEL) -> Agent:
    """Construct the coordinator and its specialists for a given model."""
    weather = Agent(
        name="weather_agent",
        model=model,
        description="Answers questions about current weather conditions in a city.",
        instruction=(
            "You are a weather specialist. Use the get_weather tool. If it "
            "returns status 'error', relay the error_message rather than guessing."
        ),
        tools=[get_weather],
    )
    time_specialist = Agent(
        name="time_agent",
        model=model,
        description="Answers questions about the current local time in a city.",
        instruction=(
            "You are a timekeeping specialist. Use the get_time tool. If it "
            "returns status 'error', relay the error_message rather than guessing."
        ),
        tools=[get_time],
    )
    return Agent(
        name="coordinator",
        model=model,
        description="Routes user requests to the correct specialist sub-agent.",
        instruction=(
            "You are a coordinator. You do not answer questions yourself. "
            "Delegate weather questions to weather_agent and time questions to "
            "time_agent. If neither applies, say the request is out of scope."
        ),
        sub_agents=[weather, time_specialist],
    )


## 2. The ADK session and runner belong in st.session_state

Streamlit re-runs the script on every interaction. Creating a fresh ADK session each time would silently reset the conversation, so both the runner and the session id must be cached. The `if "runner" not in st.session_state` guard is essential — without it every keystroke would rebuild the agents and wipe the conversation.

`create_session` is async while the Streamlit script is synchronous, so bridge with `asyncio.run`.


## 3. Streaming, routing display and the model switcher

You do **not** need to re-send the conversation history: the ADK session service already holds it, because every call reuses the same `session_id`. This is a genuine convenience difference from the OpenAI SDK, where you pass the accumulated input list yourself.

`event.author` gives the responding agent's name. Requests should show `weather_agent` or `time_agent`, not `coordinator` — if you see `coordinator` answering directly, its instruction is not firm enough about delegating.

Latency is recorded so the model trade-off is measured rather than guessed.


## 4. Write the full app


In [ ]:
%%writefile gemini_app.py
"""Streamlit front end for the Lab 08 Gemini multi-agent team."""

import asyncio
import time

import streamlit as st
from dotenv import load_dotenv
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

from gemini_team import APP_NAME, USER_ID, build_coordinator

load_dotenv()

st.set_page_config(page_title="Gemini Multi-Agent Assistant", page_icon="G")
st.title("Gemini Multi-Agent Assistant")
st.caption("Coordinator routing to weather and time specialists.")

MODELS = ["gemini-2.0-flash", "gemini-2.0-flash-lite", "gemini-2.5-flash"]


def init_backend(model: str) -> None:
    """Create the ADK session service, session and runner once per model."""
    session_service = InMemorySessionService()
    session = asyncio.run(
        session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
    )
    st.session_state.session_id = session.id
    st.session_state.runner = Runner(
        agent=build_coordinator(model),
        app_name=APP_NAME,
        session_service=session_service,
    )
    st.session_state.model = model
    st.session_state.history = []


async def stream_reply(placeholder, question: str) -> tuple[str, str]:
    """Stream the coordinator's reply, returning (text, responding_agent)."""
    message = types.Content(role="user", parts=[types.Part(text=question)])

    text, author = "", ""
    async for event in st.session_state.runner.run_async(
        user_id=USER_ID,
        session_id=st.session_state.session_id,
        new_message=message,
    ):
        if event.content and event.content.parts:
            chunk = event.content.parts[0].text or ""
            if chunk:
                if event.partial:
                    text += chunk
                else:
                    text = chunk
                placeholder.markdown(text)
        if event.is_final_response():
            author = event.author
    return text, author


if "runner" not in st.session_state:
    init_backend(MODELS[0])

with st.sidebar:
    st.header("Configuration")
    chosen = st.selectbox(
        "Gemini model", MODELS, index=MODELS.index(st.session_state.model)
    )
    if chosen != st.session_state.model:
        init_backend(chosen)
        st.rerun()

    if st.button("Clear conversation"):
        init_backend(st.session_state.model)
        st.rerun()

    st.metric("Turns", len(st.session_state.history) // 2)
    if "last_latency" in st.session_state:
        st.metric("Last reply (s)", f"{st.session_state.last_latency:.2f}")

# Repaint the conversation on every re-run.
for message in st.session_state.history:
    with st.chat_message(message["role"]):
        if message.get("agent"):
            st.caption(f"Handled by: {message['agent']}")
        st.markdown(message["content"])

prompt = st.chat_input("Ask about weather or time...")

if prompt:
    st.session_state.history.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        placeholder = st.empty()
        try:
            start = time.perf_counter()
            answer, author = asyncio.run(stream_reply(placeholder, prompt))
            st.session_state.last_latency = time.perf_counter() - start
        except Exception as exc:
            answer, author = f"The agent failed: {exc}", "error"
            placeholder.markdown(answer)
        st.caption(f"Handled by: {author}")

    st.session_state.history.append(
        {"role": "assistant", "content": answer, "agent": author}
    )


## 5. Run both apps side by side

Locally, on different ports so both are open at once:

```bash
streamlit run app.py --server.port 8501          # Lab 07, OpenAI
streamlit run gemini_app.py --server.port 8502   # this lab, Gemini
```

Send the identical set of requests to both and record what you observe:

- **Routing quality** — does each request reach the intended specialist? Test ambiguous requests that could plausibly go either way.
- **Latency** — time to first token, and time to complete reply.
- **Developer experience** — lines of setup code, how state is handled, how clear the errors were.
- **Failure behaviour** — what each does with an unknown city or an out-of-scope question.

Switching to the `-lite` variant should measurably reduce latency; check whether routing quality holds on ambiguous requests.


## 6. Record your comparison

Fill in every row from your own runs. **Fabricated numbers are not assessable evidence**; a comparison with real measurements and an honest conclusion is. In the repo folder this template is `COMPARISON.md`.


In [ ]:
%%writefile COMPARISON.md
# Multi-Agent SDK Comparison

Same architecture (routing coordinator over narrow specialists) built twice.

## Test requests
1. "What is the weather in Singapore?"    (expect weather specialist)
2. "What time is it in Tokyo?"            (expect time specialist)
3. "What is the weather on Mars?"         (expect graceful error)
4. "Tell me about Singapore."             (ambiguous - observe routing)

## Results

| Criterion | OpenAI Agents SDK | Google ADK |
|---|---|---|
| Correct routing (of 4) | _/4 | _/4 |
| Median reply latency | _ s | _ s |
| Lines of setup code | _ | _ |
| Conversation state | Passed in as input list | Held by SessionService |
| Who answered | `result.last_agent.name` | `event.author` |
| Structured output | `output_type=` | `output_schema=` |
| Built-in tracing | Yes, hosted dashboard | Yes, via `adk web` |

## Findings
- Routing: ...
- Latency: ...
- Developer experience: ...
- Failure handling: ...

## Conclusion
Use the OpenAI Agents SDK when ...
Use Google ADK when ...


## What you learned

- The ADK runner and session id must live in `st.session_state`, or Streamlit's re-run silently resets the conversation on every keystroke.
- ADK's session service holds conversation history server-side, so you do not re-send the transcript — a real difference from the OpenAI SDK's input list.
- Bridging ADK's async generator into synchronous Streamlit is done with `asyncio.run`.
- A factory function for agent construction is what makes runtime model switching possible without parent-reassignment errors.
- An SDK comparison is only evidence if the numbers are measured from your own runs.
